# Reading from the Bronze Layer

In [0]:
bronze_df = spark.table("workspace.bronze.crm_prd_info")

#Init

In [0]:
from pyspark.sql.functions import col, trim, when
from pyspark.sql.types import StringType

In [0]:
rename_map = {
    'prd_id': 'product_id',
    'prd_key': 'product_key',
    'prd_nm': 'product_name',
    'prd_cost': 'product_cost',
    'prd_line': 'product_line',
    'prd_start_dt': 'product_start_date',
    'prd_end_dt': 'product_end_date'
}



# Data Transformations

## Automatically scan and trim whitespaces from all text columns

In [0]:
for field in bronze_df.schema.fields:
    if isinstance(field.dataType, StringType):
        bronze_df = bronze_df.withColumn(field.name, trim(col(field.name)))


## Define a column renaming mapping for standard business terms

In [0]:
rename_map = {
    'prd_id': 'product_id',
    'prd_key': 'product_key',
    'prd_nm': 'product_name',
    'prd_cost': 'product_cost',
    'prd_line': 'product_line',
    'prd_start_dt': 'product_start_date',
    'prd_end_dt': 'product_end_date'
}

## Renaming cryptic columns using a translation dictionary 

In [0]:
for oldname, newname in rename_map.items():
    if oldname in bronze_df.columns:
        bronze_df = bronze_df.withColumnRenamed(oldname, newname)

## Normalizing abbreviations to descriptive business values

In [0]:
bronze_df = (
    bronze_df
    
    .withColumn(
    "product_line",
    when(col("product_line") == "R", "Road")
    .when(col("product_line") == "S", "Sport")
    .when(col("product_line") == "M", "Mountain")
    .otherwise("n/a")
    )
)


# Writing to the silver layer

In [0]:
bronze_df.write \
.mode('overwrite') \
.format('delta') \
.saveAsTable('silver.crm_products')